# I. MNIST 데이터셋 GPU 가속 학습

## $1)$ 딥러닝 학습 구조 복습

앞선 MNIST 손글씨 분류 실습에서 확인한 바와 같이, 하나의 Batch 학습 과정은 다음과 같습니다.

1. 이미지와 Label 준비
2. 순전파로 결과 예측
3. Loss 계산
4. 기존 Gradient 초기화
5. 역전파 수행
6. Optimizer로 Weight 수정

## $2)$ CPU 기반 MNIST 데이터셋 학습

MNIST 실습에서 사용된 코드를 종합하면 아래와 같이 작성할 수 있습니다.

In [1]:
import time
import random

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA 빌드:", torch.version.cuda)
print("CUDA 사용 가능:", torch.cuda.is_available())
print("GPU 개수:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA를 사용할 수 없습니다.")

PyTorch: 2.7.1+cu126
CUDA 빌드: 12.6
CUDA 사용 가능: True
GPU 개수: 1


In [ ]:
SEED = 42
DATA_ROOT = "src/datasets"

BATCH_SIZE = 64
TEST_BATCH_SIZE = 1000
EPOCHS = 3
LEARNING_RATE = 0.001
NUM_WORKERS = 0


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1  = nn.Conv2d( in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1, )
        self.conv2  = nn.Conv2d( in_channels=8, out_channels=16, kernel_size=3, stride=1, padding=1, )
        self.pool   = nn.MaxPool2d( kernel_size=2, stride=2, )
        self.fc1    = nn.Linear( 16 * 7 * 7, 64, )
        self.fc2    = nn.Linear( 64, 10, )

    def forward(self, x, return_features=False):
        conv1_output = self.conv1(x)
        relu1_output = F.relu(conv1_output)
        pool1_output = self.pool(relu1_output)
        conv2_output = self.conv2(pool1_output)
        relu2_output = F.relu(conv2_output)
        pool2_output = self.pool(relu2_output)
        flattened = torch.flatten(pool2_output, start_dim=1)
        fc1_output = F.relu(self.fc1(flattened))
        logits = self.fc2(fc1_output)

        if return_features:
            features = {
                "conv1": conv1_output, "relu1": relu1_output, "pool1": pool1_output,
                "conv2": conv2_output, "relu2": relu2_output, "pool2": pool2_output,
            }
            return logits, features

        return logits


def train_one_epoch(model, data_loader, criterion, optimizer, device):
    # 모델을 학습 모드(training mode)로 변경
    model.train()

    # 결과를 누적하기 위한 변수
    total_loss = 0.0
    correct_count = 0
    sample_count = 0

    # 학습 데이터를 DataLoader(train_loader)의 Batch 단위로 불러와 학습 반복
    for images, labels in data_loader:
        # 장치(CPU/GPU)로 이동
        images = images.to(device)
        labels = labels.to(device)

        # Gradient 초기화 (이전 Batch에서 계산한 Gradient를 제거)
        optimizer.zero_grad()

        # 순전파를 통해 원시 출력 저장 (클래스별 점수 10개)
        logits = model(images)

        # Loss 계산
        loss = criterion(logits, labels)
        # 역전파를 통한 Gradient 계산
        loss.backward()
        # Gradient를 사용하여 파라미터 수정
        optimizer.step()

        # Batch 한 개의 이미지 개수
        batch_size = images.size(0)

        # 결과 누적
        total_loss += loss.item() * batch_size                 # Loss 누적 (1개 batch의 평균 loss x batch 크기)
        predictions = logits.argmax(dim=1)                     # 가장 높은 점수를 가진 클래스를 선택
        correct_count += (predictions == labels).sum().item()  # 정답을 맞춘 개수 계산
        sample_count += batch_size                             # 전체 샘플 수 누적

    # 평균 Loss 및 Accuracy 계산
    average_loss = total_loss / sample_count
    accuracy = correct_count / sample_count

    return average_loss, accuracy


def evaluate(model, data_loader, criterion, device):
    # 모델을 평가 모드(evaluation mode)로 변경
    model.eval()

    # 결과를 누적하기 위한 변수
    total_loss = 0.0
    correct_count = 0
    sample_count = 0

    # Gradient 계산을 비활성화 (파라미터 수정 비활성화)
    with torch.no_grad():
        # 평가 데이터를 DataLoader(test_loader)의 Batch 단위로 불러와 평가 반복
        for images, labels in data_loader:
            # 장치(CPU/GPU)로 이동
            images = images.to(device)
            labels = labels.to(device)

            # 학습된 모델의 순전파를 통해 원시 출력 저장 (클래스별 점수 10개)
            logits = model(images)

            # Loss 계산 (역전파는 수행하지 않음)
            loss = criterion(logits, labels)

            # Batch 한 개의 이미지 개수
            batch_size = images.size(0)

            # 결과 누적
            total_loss += loss.item() * batch_size                 # Loss 누적 (1개 batch의 평균 loss x batch 크기)
            predictions = logits.argmax(dim=1)                     # 가장 높은 점수를 가진 클래스를 선택
            correct_count += (predictions == labels).sum().item()  # 정답을 맞춘 개수 계산
            sample_count += batch_size                             # 전체 샘플 수 누적

        # 평균 Loss 및 Accuracy 계산
        average_loss = total_loss / sample_count
        accuracy = correct_count / sample_count

        return average_loss, accuracy


def main_cpu():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    transform = transforms.ToTensor()

    train_dataset = datasets.MNIST( root=DATA_ROOT, train=True, transform=transform, download=True, )
    test_dataset  = datasets.MNIST( root="src/datasets", train=False, transform=transform, download=True, )
    train_loader  = DataLoader( train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, )
    test_loader   = DataLoader( test_dataset, batch_size=TEST_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, )

    device = torch.device("cpu")

    model = SimpleCNN()
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
    )

    train_loss_history = []
    train_accuracy_history = []
    test_loss_history = []
    test_accuracy_history = []

    training_start_time = time.perf_counter()

    for epoch in range(EPOCHS):
        epoch_start_time = time.perf_counter()

        train_loss, train_accuracy = train_one_epoch( model, train_loader, criterion, optimizer, device, )
        test_loss, test_accuracy = evaluate( model, test_loader, criterion, device, )

        train_loss_history.append(train_loss)
        train_accuracy_history.append(train_accuracy)
        test_loss_history.append(test_loss)
        test_accuracy_history.append(test_accuracy)

        epoch_time = time.perf_counter() - epoch_start_time

        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"Train loss: {train_loss:.4f} | "
            f"Train accuracy: "
            f"{train_accuracy * 100:.2f}% | "
            f"Test loss: {test_loss:.4f} | "
            f"Test accuracy: "
            f"{test_accuracy * 100:.2f}% | "
            f"Time: {epoch_time:.2f}s"
        )

    total_training_time = time.perf_counter() - training_start_time
    print(f"Total CPU training time: {total_training_time:.2f}s")

    return total_training_time

## $3)$ GPU 기반 MNIST 데이터셋 학습

CPU 기반 MNIST 데이터셋 학습 코드를 복습하며 GPU 기반 학습 코드를 작성해봅시다.

`CuPy` 실습에서 GPU 연산 처리 시간을 측정하기 위해 GPU의 모든 작업이 완료될 때까지 CPU가 기다리도록 하는 동기화 함수인 `cp.cuda.Stream.null.synchronize()` 대신, `PyTorch`에서는 `torch.cuda.synchronize()` 함수를 사용합니다.

In [4]:
# TODO: GPU 기반 MNIST 데이터셋 학습 코드 작성

# def main_gpu():
#     pass

def main_gpu():
    seed = 42

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    transform       = transforms.ToTensor()
    train_dataset   = datasets.MNIST( root="src/datasets",  train=True,     transform=transform,    download=True, )
    test_dataset    = datasets.MNIST( root="src/datasets",  train=False,    transform=transform,    download=True, )
    train_loader    = DataLoader    ( train_dataset,        batch_size=64,  shuffle=True,           num_workers=0, )
    test_loader     = DataLoader    ( test_dataset,         batch_size=1000,shuffle=False,          num_workers=0, )

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA를 사용할 수 없습니다.")

    device = torch.device("cuda")

    model = SimpleCNN()
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001,
    )

    epochs = 3

    train_loss_history = []
    train_accuracy_history = []
    test_loss_history = []
    test_accuracy_history = []

    torch.cuda.synchronize()
    training_start_time = time.perf_counter()

    for epoch in range(epochs):
        torch.cuda.synchronize()
        epoch_start_time = time.perf_counter()

        train_loss, train_accuracy = train_one_epoch(   model, train_loader, criterion, optimizer, device, )
        test_loss , test_accuracy  = evaluate(          model, test_loader , criterion,            device, )

        train_loss_history.append(train_loss)
        train_accuracy_history.append(train_accuracy)
        test_loss_history.append(test_loss)
        test_accuracy_history.append(test_accuracy)

        torch.cuda.synchronize()
        epoch_time = time.perf_counter() - epoch_start_time

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train loss: {train_loss:.4f} | "
            f"Train accuracy: "
            f"{train_accuracy * 100:.2f}% | "
            f"Test loss: {test_loss:.4f} | "
            f"Test accuracy: "
            f"{test_accuracy * 100:.2f}% | "
            f"Time: {epoch_time:.2f}s"
        )

    torch.cuda.synchronize()
    total_training_time = time.perf_counter() - training_start_time
    print(f"Total GPU training time: {total_training_time:.2f}s")

    return total_training_time

**데이터셋 학습 및 결과 비교:**

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

사용 장치: cuda


In [6]:
if __name__ == "__main__":

    print("\n=== GPU Training ===")
    gpu_training_time = main_gpu()

    print("=== CPU Training ===")
    cpu_training_time = main_cpu()

    speedup = cpu_training_time / gpu_training_time
    time_difference = cpu_training_time - gpu_training_time

    print("\n=== CPU vs. GPU Comparison ===")
    print(f"CPU training time: {cpu_training_time:.2f}s")
    print(f"GPU training time: {gpu_training_time:.2f}s")
    print(f"Time difference: {time_difference:.2f}s")
    print(f"GPU speedup: {speedup:.2f}x")


=== GPU Training ===
Epoch 1/3 | Train loss: 0.3132 | Train accuracy: 90.54% | Test loss: 0.0964 | Test accuracy: 96.92% | Time: 15.13s
Epoch 2/3 | Train loss: 0.0923 | Train accuracy: 97.12% | Test loss: 0.0702 | Test accuracy: 97.69% | Time: 15.59s
Epoch 3/3 | Train loss: 0.0669 | Train accuracy: 97.91% | Test loss: 0.0495 | Test accuracy: 98.19% | Time: 15.54s
Total GPU training time: 46.26s
=== CPU Training ===
Epoch 1/3 | Train loss: 0.3132 | Train accuracy: 90.53% | Test loss: 0.0964 | Test accuracy: 96.93% | Time: 17.38s
Epoch 2/3 | Train loss: 0.0923 | Train accuracy: 97.12% | Test loss: 0.0695 | Test accuracy: 97.77% | Time: 17.72s
Epoch 3/3 | Train loss: 0.0665 | Train accuracy: 97.92% | Test loss: 0.0494 | Test accuracy: 98.21% | Time: 17.40s
Total CPU training time: 52.50s

=== CPU vs. GPU Comparison ===
CPU training time: 52.50s
GPU training time: 46.26s
Time difference: 6.24s
GPU speedup: 1.13x
